
# Information Gain Analytics & Visualization — **Operational Reference**

Turn merged JSON into **signals** that drive scheduling: ΔInfo per scrape, cumulative curves,
half-life, dormancy, subreddit priors, and visualizations.


## 1. Load & Traverse

In [ ]:

import os, json, re
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def parse_iso_utc(s: str):
    if not s: return None
    s = s.replace('Z','+00:00')
    try:
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None: dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    except Exception:
        m = re.search(r"(\d{8})[T_]?(\d{6})", s)
        if m:
            return datetime.strptime(m.group(1)+m.group(2), "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)
        return None

def collect_nodes(data):
    nodes = []
    nodes.append(("post", data.get("post", {})))
    queue = list(data.get("comments", []) or [])
    while queue:
        c = queue.pop(0)
        nodes.append(("comment", c))
        for ch in c.get("replies", []) or []:
            if isinstance(ch, dict): queue.append(ch)
    return nodes


## 2. Compute ΔInfo Timeline

In [ ]:

def compute_delta_info(merged):
    nodes = collect_nodes(merged)
    root_scores = nodes[0][1].get("score_history", [])
    timeline = sorted(set([parse_iso_utc(ts) for _, ts in root_scores if parse_iso_utc(ts)]))
    per_ts = {t: {"new_comments":0,"edited_comments":0,"score_changed":0,"deleted_comments":0} for t in timeline}
    for idx, (kind, n) in enumerate(nodes):
        sh = n.get("score_history", [])
        if sh:
            seq = [(val, parse_iso_utc(ts)) for val, ts in sh if parse_iso_utc(ts)]
            seq.sort(key=lambda x: x[1])
            if idx!=0 and seq:
                first_ts = seq[0][1]
                if first_ts in per_ts: per_ts[first_ts]["new_comments"] += 1
            prev = None
            for val, ts in seq:
                if ts in per_ts and prev is not None and val is not None and prev is not None and val != prev:
                    per_ts[ts]["score_changed"] += 1
                prev = val
        th = n.get("text_history", [])
        for _, ts in th or []:
            t = parse_iso_utc(ts)
            if t in per_ts: per_ts[t]["edited_comments"] += 1
        if n.get("deleted") and n.get("deleted_at"):
            t = parse_iso_utc(n["deleted_at"])
            if t in per_ts: per_ts[t]["deleted_comments"] += 1
    return per_ts, timeline


## 3. Half-life, Dormancy, Waste

In [ ]:

def half_life_hours(timeline, deltas):
    if not deltas or sum(deltas)==0: return None
    cum = np.cumsum(deltas)
    target = cum[-1]/2.0
    idx = int(np.argmax(cum>=target))
    return (timeline[idx]-timeline[0]).total_seconds()/3600.0

def time_to_dormancy_hours(timeline, deltas, window=3):
    for i in range(0, len(deltas)-window+1):
        if all(v==0 for v in deltas[i:i+window]):
            return (timeline[i]-timeline[0]).total_seconds()/3600.0
    return None

def waste_rate(deltas):
    if not deltas: return None
    useful = sum(1 for x in deltas if x>0)
    return 1 - useful/len(deltas)


## 4. Subreddit Priors (λ estimation)

In [ ]:

def estimate_lambda_from_curve(timeline, deltas, eps=1e-6):
    # naive fit: ΔInfo ~ A * exp(-λ t)
    if len(timeline)<3 or sum(deltas)<=0: return None
    t_hours = np.array([(t - timeline[0]).total_seconds()/3600.0 for t in timeline])
    y = np.array(deltas, dtype=float)
    y = y / (y.max() + eps)
    y = np.clip(y, eps, 1.0)  # avoid log(0)
    coeffs = np.polyfit(t_hours, np.log(y), 1)
    lam = -coeffs[0]
    return max(lam, 0.0)

def aggregate_subreddit_priors(rows):
    out = {}
    by_sub = {}
    for r in rows:
        by_sub.setdefault(r["subreddit"], []).append(r["lambda_hat"])
    for sub, vals in by_sub.items():
        out[sub] = {
            "lambda_mean": float(sum(vals)/len(vals)),
            "lambda_median": float(np.median(vals)),
            "lambda_p80": float(np.quantile(vals, 0.8)),
            "n_posts": len(vals),
        }
    return out


## 5. Visualization Helpers

In [ ]:

def plot_delta_curve(timeline, deltas, title="ΔInfo per scrape"):
    xs = list(range(len(deltas)))
    plt.figure(); plt.plot(xs, deltas, marker='o')
    plt.title(title); plt.xlabel("Scrape index"); plt.ylabel("ΔInfo"); plt.tight_layout(); plt.show()

def plot_hist(values, title, xlabel):
    if not values: return
    plt.figure(); plt.hist(values, bins=20)
    plt.title(title); plt.xlabel(xlabel); plt.ylabel("Count"); plt.tight_layout(); plt.show()
